# MediBot — Component 4: SQL RAG over `mediassist.db`

Not every question lives in a PDF. *"How many billing claims were escalated last month?"* lives in a relational database, not a document chunk. This notebook builds `sql_rag_chain(question: str) -> str` — the same three-step shape used in `S5_01_advanced_rag.ipynb`'s SQL RAG section:

1. Translate the natural-language question into SQL, using an LLM that has seen the schema (`create_sql_query_chain`)
2. Clean the raw LLM output down to just the SQL statement (`clean_sql`)
3. Execute the SQL, then ask the LLM to turn the raw result into a natural-language answer

SQL RAG is gated to **`billing_executive`** and **`admin`** — the roles with analytical responsibilities, per the assignment. This notebook only builds/tests the chain; the actual gate is enforced by `/chat`'s routing logic in Component 5.

## 1 — Inspect the schema before building anything

Per the assignment's instruction: look at the actual tables before writing a chain that has to generate SQL against them.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "medibot"))

from langchain_community.utilities import SQLDatabase

from config import MEDIASSIST_DB_PATH

db = SQLDatabase.from_uri(f"sqlite:///{MEDIASSIST_DB_PATH}")

print("Tables:", db.get_usable_table_names())
print("\nSchema:")
print(db.get_table_info())

RAG assignment dir: /Users/vennilave23/Documents/VSCode_Workspace/CodeBasics/Assignments/2_RAG
Data dir: /Users/vennilave23/Documents/VSCode_Workspace/CodeBasics/Assignments/2_RAG/Medibot_Assignment_Resources/mediassist_data
Tables: ['claims', 'maintenance_tickets']

Schema:

CREATE TABLE claims (
	claim_id TEXT, 
	patient_id TEXT, 
	patient_name TEXT, 
	department TEXT, 
	claim_type TEXT, 
	diagnosis_code TEXT, 
	insurer TEXT, 
	claimed_amount REAL, 
	approved_amount REAL, 
	status TEXT, 
	submitted_date TEXT, 
	resolved_date TEXT, 
	PRIMARY KEY (claim_id)
)

/*
3 rows from claims table:
claim_id	patient_id	patient_name	department	claim_type	diagnosis_code	insurer	claimed_amount	approved_amount	status	submitted_date	resolved_date
CLM-2024-1000	PAT-51347	Kavya Pillai	nephrology	reimbursement	N17.9	New India Assurance	72700.0	None	pending	2024-01-26	None
CLM-2024-1001	PAT-75435	Kavya Das	cardiology	cashless	I21.4	Bajaj Allianz	129900.0	None	pending	2024-04-21	None
CLM-2024-1002	PAT-5744

/var/folders/pg/y49g43_121gbwclljyf9msgc0000gn/T/ipykernel_29856/551578340.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [2]:
import sqlite3

import pandas as pd

conn = sqlite3.connect(MEDIASSIST_DB_PATH)
df_claims = pd.read_sql("SELECT * FROM claims LIMIT 3", conn)
df_tickets = pd.read_sql("SELECT * FROM maintenance_tickets LIMIT 3", conn)
conn.close()

print(f"claims: {len(df_claims.columns)} columns")
display(df_claims)
print(f"\nmaintenance_tickets: {len(df_tickets.columns)} columns")
display(df_tickets)

claims: 12 columns


,claim_id,patient_id,patient_name,department,claim_type,diagnosis_code,insurer,claimed_amount,approved_amount,status,submitted_date,resolved_date
0,CLM-2024-1000,PAT-51347,Kavya Pillai,nephrology,reimbursement,N17.9,New India Assurance,72700.0,NaN,pending,2024-01-26,NaN
1,CLM-2024-1001,PAT-75435,Kavya Das,cardiology,cashless,I21.4,Bajaj Allianz,129900.0,NaN,pending,2024-04-21,NaN
2,CLM-2024-1002,PAT-57447,Pari Naidu,neurology,cashless,I63.9,United India,91000.0,83200.0,approved,2024-12-19,2024-12-28



maintenance_tickets: 12 columns


,ticket_id,equipment_name,equipment_id,category,campus,issue_type,fault_code,raised_by,raised_date,resolved_date,status,resolution_note
0,TKT-2024-2000,SterilPro 3000,EQ-HC-3588,sterilisation,MediAssist Hyderabad Central,preventive_maintenance,NaN,Arjun Desai,2024-11-18,NaN,in_progress,NaN
1,TKT-2024-2001,DriveFlow IP-200,EQ-HC-8484,infusion,MediAssist Hyderabad Central,sensor_failure,F-05,Riya Nair,2024-11-14,2024-11-26,resolved,"Firmware/drug library updated, verified"
2,TKT-2024-2002,DriveFlow IP-200,EQ-HC-5847,infusion,MediAssist Hyderabad Central,battery_replacement,F-01,Kiara Sharma,2024-04-13,2024-04-23,resolved,"Door seal replaced, leak test passed"


## 2 — RBAC gate: who can use SQL RAG

`is_sql_rag_permitted(role)` is a standalone check — separate from `sql_rag_chain()` itself, which only takes a question. The `/chat` endpoint (Component 5) calls this *before* routing to `sql_rag_chain()`, so a disallowed role never even reaches the database.

In [3]:
from sql_rag import is_sql_rag_permitted

for role in ["doctor", "nurse", "billing_executive", "technician", "admin"]:
    print(f"  {role:<18} permitted={is_sql_rag_permitted(role)}")

  doctor             permitted=False
  nurse              permitted=False
  billing_executive  permitted=True
  technician         permitted=False
  admin              permitted=True


## 3 — The chain, and why `clean_sql()` exists

`sql_query_chain` (LangChain's `create_sql_query_chain`) is prompted with the schema above and returns *text*, not guaranteed-clean SQL. Groq-hosted models routinely prefix it with `"Question: ...\nSQLQuery:"` — executing that raw text against sqlite3 fails, so `clean_sql()` strips it before anything touches the database.

In [4]:
from sql_rag import clean_sql, sql_query_chain

raw_sql = sql_query_chain.invoke({"question": "How many claims are pending?"})
print(f"Raw LLM output:\n  {raw_sql!r}\n")
print(f"After clean_sql():\n  {clean_sql(raw_sql)!r}")

Raw LLM output:
  'Question: How many claims are pending?  \nSQLQuery:  \n```sql\nSELECT COUNT(*) AS pending_count\nFROM "claims"\nWHERE "status" = \'pending\';\n```'

After clean_sql():
  'SELECT COUNT(*) AS pending_count\nFROM "claims"\nWHERE "status" = \'pending\';'


## 4 — `sql_rag_chain(question)` end to end

Tested on 4 different analytical questions spanning both tables, per the assignment's requirement.

In [5]:
import logging

from sql_rag import sql_rag_chain

# INFO so sql_rag.py's generated-SQL logging is visible per question.
logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("medibot.sql_rag").setLevel(logging.INFO)


def ask_sql(question: str) -> None:
    print(f"Question: {question}")
    answer = sql_rag_chain(question)
    print(f"Answer:   {answer}")
    print("-" * 70)

In [6]:
ask_sql("How many billing claims are currently pending?")

Question: How many billing claims are currently pending?


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Generated SQL: SELECT COUNT(*) AS "pending_count" FROM claims WHERE "status"='pending';


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Answer:   There are 17 billing claims currently pending.
----------------------------------------------------------------------


In [7]:
ask_sql("What is the total approved amount for claims in the cardiology department?")

Question: What is the total approved amount for claims in the cardiology department?


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Generated SQL: SELECT COALESCE(SUM("approved_amount"),0) AS "total_approved" FROM "claims" WHERE "department"='cardiology';


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Answer:   The total approved amount for claims in the cardiology department is ₹394,100.
----------------------------------------------------------------------


In [8]:
ask_sql("How many maintenance tickets are still in progress?")

Question: How many maintenance tickets are still in progress?


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Generated SQL: SELECT COUNT(*) AS in_progress_count FROM maintenance_tickets WHERE "status" = 'in_progress';


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Answer:   There are 15 maintenance tickets still in progress.
----------------------------------------------------------------------


In [9]:
ask_sql("Which equipment category has the most maintenance tickets?")

Question: Which equipment category has the most maintenance tickets?


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Generated SQL: SELECT "category", COUNT(*) AS ticket_count
FROM maintenance_tickets
GROUP BY "category"
ORDER BY ticket_count DESC
LIMIT 5;


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Answer:   The **monitoring** equipment category has the most maintenance tickets, with **25 tickets**.
----------------------------------------------------------------------


## 5 — Simulating the `/chat` routing decision

A preview of Component 5's logic: an analytical question from a permitted role goes to `sql_rag_chain`; the same question from a role without analytical access is refused *before* any SQL is generated — no query, no data, nothing to leak.

In [10]:
def route_analytical_question(question: str, role: str) -> str:
    if not is_sql_rag_permitted(role):
        return (
            f"As a {role}, you do not have access to analytics queries. "
            "I can only answer document questions from your permitted collections."
        )
    return sql_rag_chain(question)


question = "How many billing claims are currently pending?"
for role in ["nurse", "billing_executive"]:
    print(f"[{role}] {question}")
    print(f"  -> {route_analytical_question(question, role)}\n")

HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


Retrying request to /openai/v1/chat/completions in 5.000000 seconds


[nurse] How many billing claims are currently pending?
  -> As a nurse, you do not have access to analytics queries. I can only answer document questions from your permitted collections.

[billing_executive] How many billing claims are currently pending?


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


Retrying request to /openai/v1/chat/completions in 1.000000 seconds


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Generated SQL: SELECT COUNT(*) AS "pending_count" FROM claims WHERE "status"='pending';


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


  -> There are **17 billing claims currently pending**.



## 6 — Next: Component 5 (FastAPI backend)

The `/chat` endpoint composes everything built so far: RBAC-gated routing (`is_sql_rag_permitted`, Section 5 above) decides between `sql_rag_chain` (this notebook) and `hybrid_search -> rerank -> generate_answer` (Components 2-3), and returns `{answer, sources, retrieval_type, role}`.